# ALICE RL — Production TRL GRPO Training

Full real RL training loop against the live ALICE adversarial environment.

**What this does:**
- Loads a small model locally on GPU with LoRA
- Runs proper 3-turn episodes: `reset → generate → step → feedback → generate → step → ...`
- Uses GRPO (group-normalised policy gradient) with real rewards from the ALICE verifier
- Pushes every metric (rewards, advantages, loss, disc_coverage) to the live Space dashboard
- Registers this Colab session as a live job in the HF Space & Jobs tab

**Runtime:** T4 GPU — Runtime → Change runtime type → T4 GPU

**Models (change `MODEL_ID`):**
- `Qwen/Qwen2.5-0.5B-Instruct` — ~1 GB, fastest on T4 ✅
- `Qwen/Qwen2.5-1.5B-Instruct` — ~3 GB, good quality
- `HuggingFaceTB/SmolLM2-1.7B-Instruct` — ~3.5 GB
- `Qwen/Qwen2.5-3B-Instruct` — ~6 GB, use 4-bit

In [ ]:
!pip install -q httpx torch transformers peft accelerate trl bitsandbytes
print('✅ Dependencies installed')

In [ ]:
import os, uuid

# ── Configuration ─────────────────────────────────────────────────────
ALICE_ENV_URL  = 'https://rohanjain1648-alice-rl-environment.hf.space'
MODEL_ID       = 'Qwen/Qwen2.5-0.5B-Instruct'  # change to any model above
EPISODES       = 50     # 50 eps ≈ 15 min on T4; use 100+ for real training
GROUP_SIZE     = 4      # rollouts per GRPO update
MAX_TURNS      = 3      # full 3-turn episodes
LR             = 1e-5
KL_COEF        = 0.04
MAX_NEW_TOKENS = 64
LOAD_IN_4BIT   = False  # set True for 3B+ models
LORA_R         = 16
HF_TOKEN       = ''     # optional: for Gemma or Hub push

# Colab session ID (shows as job in dashboard)
JOB_ID  = str(uuid.uuid4())[:24]
JOB_URL = f'https://colab.research.google.com/  (session {JOB_ID[:8]})'

print(f'Model:    {MODEL_ID}')
print(f'Episodes: {EPISODES} | Group: {GROUP_SIZE} | Turns: {MAX_TURNS}')
print(f'Space:    {ALICE_ENV_URL}')
print(f'Job ID:   {JOB_ID}')

In [ ]:
import httpx, json

_env = httpx.Client(base_url=ALICE_ENV_URL, timeout=30.0)

h = _env.get('/health').json()
print('✅ Space healthy:')
print(json.dumps(h, indent=2))

# Register this Colab session as a live job
_env.post('/jobs/register', json={
    'job_id': JOB_ID, 'model': MODEL_ID, 'episodes': EPISODES,
    'avg_reward': 0.0, 'success_rate': 0.0, 'elapsed_s': 0.0,
    'status': 'RUNNING', 'url': JOB_URL,
    'timestamp': __import__('time').strftime('%Y-%m-%dT%H:%M:%SZ', __import__('time').gmtime()),
})
print(f'✅ Job registered in dashboard: {JOB_ID[:12]}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
) if LOAD_IN_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto',
    trust_remote_code=True, torch_dtype=torch.bfloat16,
)
if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model)

# Detect LoRA target modules
candidates = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
targets = list({c for name,_ in model.named_modules() for c in candidates if name.endswith(c)})
if not targets: targets = ['q_proj','v_proj']

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_R*2,
    target_modules=targets, lora_dropout=0.05, bias='none',
))
model.print_trainable_parameters()
print('✅ Model ready')

In [ ]:
import numpy as np

_SYSTEM = (
    'You are a precise Python solver. '
    'For every task, output ONLY: result = <value>\n'
    'No explanations. No markdown. Just: result = <value>'
)

def generate(task: str, feedback: str = '') -> str:
    messages = [{'role': 'system', 'content': _SYSTEM}]
    if feedback:
        messages += [
            {'role': 'user',      'content': f'Task: {task}'},
            {'role': 'assistant', 'content': 'result = ...'},
            {'role': 'user',      'content': f'Feedback: {feedback}\nTask: {task}'},
        ]
    else:
        messages.append({'role': 'user', 'content': f'Task: {task}'})
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt += 'result ='
    except Exception:
        prompt = f'{_SYSTEM}\n\nTask: {task}\nresult ='
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=True, temperature=0.7, top_p=0.92,
                             repetition_penalty=1.1,
                             pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    first = raw.split('\n')[0].strip()
    return first if first.startswith('result') else f'result = {first[:200]}'

def run_episode() -> dict:
    ep    = _env.post('/reset').json()
    ep_id = ep['episode_id']
    task  = ep['task']
    turn_rewards, turn_composites, turn_actions = [], [], []
    feedback = ''
    total_r  = 0.0
    for turn in range(1, MAX_TURNS + 1):
        action = generate(task, feedback)
        result = _env.post('/step', json={'episode_id': ep_id, 'action': action}).json()
        r         = float(result.get('reward', 0.01))
        done      = result.get('done', False)
        verif     = result.get('info', {}).get('verification', {})
        composite = float(verif.get('composite_score', 0.0))
        turn_rewards.append(r); turn_composites.append(composite); turn_actions.append(action)
        total_r += r
        if composite >= 0.5:
            feedback = f'Turn {turn} passed (score={composite:.2f}).'
        else:
            t1 = verif.get('tier1_details', {}) or {}
            feedback = (f'Error: {t1.get("error_message","unknown")}. Fix the code.'
                        if not t1.get('success', True)
                        else f'Score {composite:.2f}. Improve correctness.')
        if done: break
    return {'task': task, 'actions': turn_actions, 'turn_rewards': turn_rewards,
            'turn_composites': turn_composites, 'total_reward': total_r,
            'success': (turn_composites[-1] if turn_composites else 0.0) >= 0.5,
            'composite': turn_composites[-1] if turn_composites else 0.0}

def collect_rollouts():
    rewards, successes, composites, prompts, responses = [], [], [], [], []
    for _ in range(GROUP_SIZE):
        try:
            ep = run_episode()
            rewards.append(ep['total_reward']); successes.append(1.0 if ep['success'] else 0.0)
            composites.append(ep['composite']); prompts.append(ep['task'])
            responses.append(ep['actions'][-1] if ep['actions'] else 'result = None')
        except Exception as exc:
            print(f'Rollout error: {exc}')
            rewards.append(0.01); successes.append(0.0)
            composites.append(0.0); prompts.append(''); responses.append('')
    return rewards, successes, composites, prompts, responses

def grpo_update(optimizer, prompts, responses, rewards):
    device = next(model.parameters()).device
    r_t    = torch.tensor(rewards, dtype=torch.float32, device=device)
    adv    = (r_t - r_t.mean()) / (r_t.std().clamp(min=1e-6))
    total_loss = None; n = 0
    for a, prompt, resp in zip(adv, prompts, responses):
        if not prompt or not resp: continue
        full = prompt + '\nresult =' + resp.replace('result =', '').strip()
        inputs = tokenizer(full, return_tensors='pt', truncation=True, max_length=512).to(device)
        labels = inputs['input_ids'].clone()
        plen   = min(tokenizer(prompt + '\nresult =', return_tensors='pt',
                               truncation=True, max_length=512)['input_ids'].shape[1],
                     labels.shape[1] - 1)
        labels[0, :plen] = -100
        out      = model(**inputs, labels=labels)
        log_prob = -out.loss
        loss_i   = -(a * log_prob) + KL_COEF * (log_prob ** 2)
        total_loss = loss_i if total_loss is None else total_loss + loss_i
        n += 1
    if n == 0 or total_loss is None: return 0.0, [0.0]*len(rewards)
    total_loss = total_loss / n
    optimizer.zero_grad(); total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return float(total_loss.detach()), adv.cpu().tolist()

print('✅ Helpers ready')

In [ ]:
import time

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPISODES, eta_min=LR*0.1)

all_rewards, all_successes, all_composites, all_losses = [], [], [], []
cumul = []
t0 = time.time()

for ep in range(1, EPISODES + 1):
    rewards, successes, composites, prompts, responses = collect_rollouts()
    loss, advantages = grpo_update(optimizer, prompts, responses, rewards)
    scheduler.step()

    all_rewards.extend(rewards); all_successes.extend(successes)
    all_composites.extend(composites); all_losses.append(loss)
    ep_mean_r = float(np.mean(rewards))
    cumul.append(ep_mean_r)

    W = 80
    avg_r    = float(np.mean(all_rewards[-W:]))
    avg_succ = float(np.mean(all_successes[-W:]))
    disc     = float(np.mean([1.0 if 0.2 < c < 0.8 else 0.0 for c in all_composites[-W:]]))
    elapsed  = time.time() - t0

    if ep % 5 == 0:
        print(f'Ep {ep:3d}/{EPISODES} | loss={loss:.4f} | ep_r={ep_mean_r:.4f} | '
              f'avg_r={avg_r:.4f} | succ={avg_succ:.0%} | disc={disc:.0%} | {elapsed:.0f}s')

    # Push all metrics to Space dashboard
    try:
        _env.post('/training/push', json={
            'model_id': MODEL_ID, 'episode': ep,
            'rewards': [round(float(r),4) for r in rewards],
            'advantages': [round(float(a),4) for a in advantages],
            'loss': round(float(loss),6),
            'success_rate': round(avg_succ,4),
            'disc_coverage': round(disc,4),
            'composites': [round(float(c),4) for c in composites],
            'cumulative_rewards': [round(float(r),4) for r in cumul],
        })
    except Exception: pass

    # Update job status every 10 episodes
    if ep % 10 == 0:
        try:
            _env.post('/jobs/register', json={
                'job_id': JOB_ID, 'model': MODEL_ID, 'episodes': EPISODES,
                'avg_reward': round(avg_r,4), 'success_rate': round(avg_succ,4),
                'elapsed_s': round(elapsed,1), 'status': 'RUNNING', 'url': JOB_URL,
                'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            })
        except Exception: pass

elapsed = time.time() - t0
final_r    = float(np.mean(all_rewards))
final_succ = float(np.mean(all_successes))
print(f'\n✅ Training complete in {elapsed:.0f}s')
print(f'avg_reward={final_r:.4f} | success={final_succ:.0%} | episodes={len(all_rewards)}')

# Mark job as completed
_env.post('/jobs/register', json={
    'job_id': JOB_ID, 'model': MODEL_ID, 'episodes': EPISODES,
    'avg_reward': round(final_r,4), 'success_rate': round(final_succ,4),
    'elapsed_s': round(elapsed,1), 'status': 'COMPLETED', 'url': JOB_URL,
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
})
print('✅ Job marked COMPLETED in dashboard')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# 1. Reward curve
ax1 = fig.add_subplot(gs[0, 0])
W = 5
ax1.plot(cumul, alpha=0.4, color='#4a90e2', linewidth=0.8, label='per-episode')
if len(cumul) >= W:
    ma = np.convolve(cumul, np.ones(W)/W, mode='valid')
    ax1.plot(range(W-1, len(cumul)), ma, color='#4a90e2', linewidth=2.5, label=f'{W}-ep MA')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Reward Curve', fontweight='bold'); ax1.set_xlabel('Episode'); ax1.legend(fontsize=8)

# 2. Reward distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(all_rewards, bins=25, color='#4a90e2', edgecolor='white', alpha=0.85)
ax2.axvline(np.mean(all_rewards), color='red', linestyle='--', label=f'mean={np.mean(all_rewards):.3f}')
ax2.set_title('Reward Distribution', fontweight='bold'); ax2.legend(fontsize=8)

# 3. Success rate
ax3 = fig.add_subplot(gs[1, 0])
succ_ep = [float(np.mean(all_successes[max(0,i-GROUP_SIZE):i+GROUP_SIZE]))
           for i in range(0, len(all_successes), GROUP_SIZE)]
ax3.plot(succ_ep, color='#27ae60', linewidth=2)
ax3.fill_between(range(len(succ_ep)), succ_ep, alpha=0.15, color='#27ae60')
ax3.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='50% target')
ax3.set_ylim(0, 1); ax3.set_title('Success Rate', fontweight='bold'); ax3.legend(fontsize=8)

# 4. Training loss
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(all_losses, color='#e74c3c', linewidth=1.5, alpha=0.7)
if len(all_losses) >= W:
    ma_l = np.convolve(all_losses, np.ones(W)/W, mode='valid')
    ax4.plot(range(W-1, len(all_losses)), ma_l, color='#c0392b', linewidth=2.5)
ax4.set_title('Training Loss (GRPO)', fontweight='bold')

fig.suptitle(f'ALICE RL — {MODEL_ID.split("/")[-1]} | {EPISODES} episodes | '
             f'avg_reward={final_r:.4f} | success={final_succ:.0%}',
             fontsize=13, fontweight='bold')
plt.savefig('/content/alice_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to /content/alice_training.png')

In [ ]:
CKPT = f'/content/{MODEL_ID.replace("/","_")}_alice_ep{EPISODES}'
model.save_pretrained(CKPT)
tokenizer.save_pretrained(CKPT)
print(f'✅ Checkpoint saved to {CKPT}')

# Check live leaderboard
lb = _env.get('/leaderboard').json()
print('\nLeaderboard:')
for e in lb:
    print(f"  #{e['rank']} {e.get('display_name',e['model_id'].split('/')[-1]):25s} "
          f"rl={e['rl_score']:.4f} reward={e['avg_reward']:.4f} ep={e['episodes_run']}")